In [ ]:
# ===========================
# Cell 1: Install dependencies
# ===========================
!pip uninstall -y torch torchvision torchaudio -q

!pip install -q torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 accelerate==0.34.2 datasets==3.0.2
!pip install -q peft==0.12.0 bitsandbytes==0.43.3 trl==0.11.5 safetensors==0.4.3


ERROR: Could not find a version that satisfies the requirement trl==0.11.5 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.1.0, 0.2.0, 0.2.1, 0.3.0, 0.3.1, 0.4.0, 0.4.1, 0.4.2, 0.4.3, 0.4.4, 0.4.5, 0.4.6, 0.4.7, 0.5.0, 0.6.0, 0.7.0, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.7.5, 0.7.6, 0.7.7, 0.7.8, 0.7.9, 0.7.10, 0.7.11, 0.8.0, 0.8.1, 0.8.2, 0.8.3, 0.8.4, 0.8.5, 0.8.6, 0.9.2, 0.9.3, 0.9.4, 0.9.6, 0.10.1, 0.11.0, 0.11.1, 0.11.2, 0.11.3, 0.11.4, 0.12.0, 0.12.1, 0.12.2, 0.13.0, 0.14.0, 0.15.0, 0.15.1, 0.15.2, 0.16.0, 0.16.1, 0.17.0, 0.18.0, 0.18.1, 0.18.2, 0.19.0, 0.19.1, 0.20.0, 0.21.0)
ERROR: No matching distribution found for trl==0.11.5


In [1]:
!pip install -U pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118   # pick right CUDA wheel
!pip install transformers accelerate datasets peft bitsandbytes safetensors
# Optional: trl for SFTTrainer and RL components
!pip install trl
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -U bitsandbytes
!pip install -U transformers peft accelerate



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu118
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 185.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 48.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 193.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 41.7 MB/s  0:00:10
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 34.6 MB/s  0:00:08
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 59.0 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 52.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import userdata
import os

# Make sure the key name matches exactly what you saved
HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN is None:
    raise RuntimeError("HF_TOKEN not found in Colab secrets. Check the 🔑 Secrets tab.")

os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
print("Token loaded successfully ✅")


Token loaded successfully ✅


In [3]:
from huggingface_hub import login
login(HF_TOKEN)


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import pandas as pd
import numpy as np
from datasets import Dataset


In [7]:
# Load dataset
dataset_path = "/content/product_review.csv"
if not os.path.exists(dataset_path):
    dataset_path = "/mnt/data/product_review.csv"
df = pd.read_csv(dataset_path)

# Ensure columns
if "review" not in df.columns or "label" not in df.columns:
    raise ValueError("CSV must contain 'review' and 'label' columns")

# Map labels
def map_label(v):
    s = str(v).strip().lower()
    if s in ["positive","pos","+","p"]: return "positive"
    if s in ["negative","neg","-","n"]: return "negative"
    if s in ["neutral","neu","0"]: return "neutral"
    try:
        v = float(s)
        if v >= 4: return "positive"
        elif v == 3: return "neutral"
        else: return "negative"
    except: return s

df["label"] = df["label"].apply(map_label)
dataset = Dataset.from_pandas(df[["review","label"]]).shuffle(seed=42)
print(dataset[0])


{'review': 'terrific purchase', 'label': 'positive'}


In [10]:
# ===========================
# Cell 3: Load Model & Tokenizer with QLoRA
# ===========================
MODEL_ID = "NousResearch/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [11]:
# ===========================
# Cell 4: Tokenization
# ===========================
def make_prompt(text):
    return f"### Instruction:\nClassify the sentiment of the following product review.\n\nReview: {text}\n\nAnswer:"

label_map = {"positive":" Positive","negative":" Negative","neutral":" Neutral"}

def tokenize_fn(ex):
    prompt = make_prompt(ex["review"])
    answer = label_map.get(ex["label"], " " + ex["label"])
    full = prompt + answer
    enc_full = tokenizer(full, truncation=True, max_length=512)
    enc_prompt = tokenizer(prompt, truncation=True, max_length=512)
    labels = enc_full["input_ids"].copy()
    for i in range(len(enc_prompt["input_ids"])):
        labels[i] = -100
    return {"input_ids": enc_full["input_ids"], "attention_mask": enc_full["attention_mask"], "labels": labels}

tokenized = dataset.map(tokenize_fn, remove_columns=dataset.column_names)


Map:   0%|          | 0/205052 [00:00<?, ? examples/s]

In [13]:
!pip install -q --upgrade transformers accelerate datasets


In [15]:
import transformers
print(transformers.__version__)


4.55.2


In [19]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./llama2_qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    do_eval=True,
    eval_steps=200,
)


In [21]:
# ===========================
# Cell 5: Training
# ===========================
from transformers import Trainer, TrainingArguments, default_data_collator

split = tokenized.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]

args = TrainingArguments(
    output_dir="./llama2_qlora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    max_steps=200,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    do_eval=True,
    eval_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    report_to="none",
    optim="paged_adamw_32bit",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)
trainer.train()
trainer.save_model("./llama2_qlora")
tokenizer.save_pretrained("./llama2_qlora")


/tmp/ipython-input-1634909520.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.161300
20,0.178400
30,0.191600
40,0.146400
50,0.201500
60,0.158800
70,0.262500
80,0.213600
90,0.211300
100,0.224500


('./llama2_qlora/tokenizer_config.json',
 './llama2_qlora/special_tokens_map.json',
 './llama2_qlora/tokenizer.model',
 './llama2_qlora/added_tokens.json')

In [22]:
# ===========================
# Cell 6: Reload Model for Inference
# ===========================
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
model = PeftModel.from_pretrained(base, "./llama2_qlora")
model.eval()
tokenizer = AutoTokenizer.from_pretrained("./llama2_qlora", use_fast=False)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [23]:
# ===========================
# Cell 7: Testing (Enter Review)
# ===========================
def predict_sentiment(text):
    prompt = make_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gen_low = gen.lower()
    if "positive" in gen_low: return "positive"
    if "negative" in gen_low: return "negative"
    if "neutral" in gen_low: return "neutral"
    return gen

# Interactive test
review = input("Enter a product review: ")
print("Predicted sentiment:", predict_sentiment(review))


Enter a product review: very good product , i like it


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Predicted sentiment: positive
